# Create Staged Invoices

Use this notebook for a project with one upfront deposit and one final outstanding-balance invoice. It creates invoices and records a payment only after you have independently confirmed it. It does not process bank transactions.

## Private Project Record

Create one private JSON record per project from `data/staged_project.example.json`, place it under `data/staged_projects/`, and configure its path below. The record persists invoice and confirmed-payment history across notebook restarts. It is ignored by Git.

In [ ]:
import os
from pathlib import Path

from IPython.display import Markdown, display

from invoice_generator import (
    create_deposit_invoice,
    create_final_balance_invoice,
    create_receipt,
    format_currency,
    load_business_config,
    load_staged_project,
    record_deposit_payment,
    record_final_balance_payment,
)
from invoice_generator.loader import PROJECT_ROOT

# Set one operation: summary, create_deposit, record_deposit, create_balance,
# record_balance, or create_receipt.
OPERATION = os.environ.get("INVOICE_STAGED_OPERATION", "summary")
PROJECT_RECORD_PATH = Path(
    os.environ.get(
        "INVOICE_STAGED_PROJECT_PATH",
        PROJECT_ROOT / "data" / "staged_projects" / "project.json",
    )
)
BUSINESS_PATH = Path(
    os.environ.get("INVOICE_BUSINESS_PATH", PROJECT_ROOT / "data" / "business.json")
)
DEPOSIT_INVOICE_PATH = Path(
    os.environ.get("INVOICE_DEPOSIT_INVOICE_PATH", PROJECT_ROOT / "data" / "invoice.json")
)
OUTPUT_DIR = Path(os.environ.get("INVOICE_OUTPUT_DIR", PROJECT_ROOT / "output"))
RECEIPT_DIR = Path(os.environ.get("INVOICE_RECEIPT_DIR", PROJECT_ROOT / "data" / "receipts"))
PAYMENT_RECORD_DIR = Path(
    os.environ.get("INVOICE_PAYMENT_RECORD_DIR", PROJECT_ROOT / "data" / "payment_records")
)

# Provide these only when recording a confirmed deposit.
PAYMENT_INVOICE_NUMBER = os.environ.get("INVOICE_DEPOSIT_PAYMENT_NUMBER", "")
PAYMENT_AMOUNT_RECEIVED = os.environ.get("INVOICE_DEPOSIT_PAYMENT_AMOUNT", "")
PAYMENT_RECEIVED_DATE = os.environ.get("INVOICE_DEPOSIT_PAYMENT_DATE", "")
PAYMENT_REFERENCE = os.environ.get("INVOICE_DEPOSIT_PAYMENT_REFERENCE")

# Provide these only when recording a confirmed final-balance payment.
FINAL_PAYMENT_INVOICE_NUMBER = os.environ.get("INVOICE_FINAL_PAYMENT_NUMBER", "")
FINAL_PAYMENT_AMOUNT_RECEIVED = os.environ.get("INVOICE_FINAL_PAYMENT_AMOUNT", "")
FINAL_PAYMENT_RECEIVED_DATE = os.environ.get("INVOICE_FINAL_PAYMENT_DATE", "")
FINAL_PAYMENT_REFERENCE = os.environ.get("INVOICE_FINAL_PAYMENT_REFERENCE")

# Provide an issued staged invoice number only when creating its receipt.
RECEIPT_INVOICE_NUMBER = os.environ.get("INVOICE_RECEIPT_INVOICE_NUMBER", "")

# Provide a new number and issue date only when creating the final-balance invoice.
FINAL_INVOICE_NUMBER = os.environ.get("INVOICE_FINAL_INVOICE_NUMBER", "")
FINAL_ISSUE_DATE = os.environ.get("INVOICE_FINAL_ISSUE_DATE", "")


## Validate and Review

This summary intentionally omits customer contact and banking details. Confirm the project total, deposit amount, and balance before selecting an invoice operation.

In [ ]:
project = load_staged_project(PROJECT_RECORD_PATH)
business = load_business_config(BUSINESS_PATH)
currency = business.currency

display(
    Markdown(
        f"**Project:** `{project.project_id}`  \n"
        f"**Agreed value:** {format_currency(project.project_total, currency)}  \n"
        f"**Deposit:** {project.deposit_percentage:g}% \n"
        f"**Deposit due:** {format_currency(project.deposit_amount, currency)}  \n"
        f"**Remaining balance:** {format_currency(project.remaining_balance, currency)}  \n"
        f"**Deposit status:** {project.deposit_status}  \n"
        f"**Final invoice status:** {project.final_invoice_status}"
    )
)

## Run an Operation

- `create_deposit` creates one unpaid deposit invoice from the configured private invoice template.
- `record_deposit` records a payment only after you confirm the project, invoice number, actual received amount, date, and optional reference. Partial payments remain unpaid.
- `create_balance` requires a fully recorded deposit and an explicit new invoice number and issue date. Its due date is automatically two calendar days later. It creates one unpaid final-balance invoice.
- `record_balance` records a full confirmed final-balance payment only after the final invoice exists.
- `create_receipt` creates or recovers the permanently assigned receipt for a confirmed deposit or final payment.
- `summary` changes nothing.

The renderer refuses to overwrite an existing PDF, and the project record rejects duplicate stage invoices and payments.

In [ ]:
if OPERATION == "summary":
    display(Markdown("No changes made. Select an operation when ready."))
elif OPERATION == "create_deposit":
    pdf_path = create_deposit_invoice(
        PROJECT_RECORD_PATH,
        DEPOSIT_INVOICE_PATH,
        BUSINESS_PATH,
        OUTPUT_DIR,
        PAYMENT_RECORD_DIR,
    )
    display(Markdown(f"**Unpaid deposit invoice created.**  \nOutput: `{pdf_path}`"))
elif OPERATION == "record_deposit":
    if not all([PAYMENT_INVOICE_NUMBER, PAYMENT_AMOUNT_RECEIVED, PAYMENT_RECEIVED_DATE]):
        raise ValueError("Provide the invoice number, received amount, and received date.")
    project = record_deposit_payment(
        PROJECT_RECORD_PATH,
        PAYMENT_INVOICE_NUMBER,
        PAYMENT_AMOUNT_RECEIVED,
        PAYMENT_RECEIVED_DATE,
        PAYMENT_REFERENCE,
        PAYMENT_RECORD_DIR,
    )
    display(Markdown("**Deposit payment recorded as confirmed.** No new PDF was created."))
elif OPERATION == "create_balance":
    if not all([FINAL_INVOICE_NUMBER, FINAL_ISSUE_DATE]):
        raise ValueError("Provide a new invoice number and issue date.")
    pdf_path = create_final_balance_invoice(
        PROJECT_RECORD_PATH,
        FINAL_INVOICE_NUMBER,
        FINAL_ISSUE_DATE,
        BUSINESS_PATH,
        OUTPUT_DIR,
        PAYMENT_RECORD_DIR,
    )
    display(Markdown(f"**Unpaid final-balance invoice created.**  \nOutput: `{pdf_path}`"))
elif OPERATION == "record_balance":
    if not all([
        FINAL_PAYMENT_INVOICE_NUMBER,
        FINAL_PAYMENT_AMOUNT_RECEIVED,
        FINAL_PAYMENT_RECEIVED_DATE,
    ]):
        raise ValueError("Provide the final invoice number, received amount, and received date.")
    record_final_balance_payment(
        PROJECT_RECORD_PATH,
        FINAL_PAYMENT_INVOICE_NUMBER,
        FINAL_PAYMENT_AMOUNT_RECEIVED,
        FINAL_PAYMENT_RECEIVED_DATE,
        FINAL_PAYMENT_REFERENCE,
        PAYMENT_RECORD_DIR,
    )
    display(Markdown("**Final-balance payment recorded as confirmed.** No new PDF was created."))
elif OPERATION == "create_receipt":
    if RECEIPT_INVOICE_NUMBER not in {
        project.deposit_invoice_number,
        project.final_invoice_number,
    }:
        raise ValueError("Provide an issued deposit or final invoice number for the receipt.")
    receipt_path = create_receipt(PAYMENT_RECORD_DIR, RECEIPT_INVOICE_NUMBER, RECEIPT_DIR)
    display(Markdown(f"**Receipt created.**  \nOutput: `{receipt_path}`"))
else:
    raise ValueError(
        "OPERATION must be summary, create_deposit, record_deposit, "
        "create_balance, record_balance, or create_receipt."
    )